In [ ]:
# In[1]:

import pandas as pd

# Load CSV files
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# 1) metric.csv: unique (cmdb_id, kpi_name) pairs with global P95 and total count
metric_grp = df_metric.groupby(["cmdb_id", "kpi_name"])["value"]
metric_summary = (
    metric_grp.agg(p95=lambda x: float(x.quantile(0.95)), count_total="count")
    .reset_index()
    .sort_values(["cmdb_id", "kpi_name"])
    .head(50)
)

# 2) trace.csv: unique trace_name values with counts and sample cmdb_ids (up to 5)
def sample_ids(series, limit=5):
    u = pd.unique(series.dropna())
    return ",".join(u[:limit].astype(str))

trace_summary = (
    df_trace.groupby("trace_name")
    .agg(count=("trace_name", "size"), sample_cmdb_ids=("cmdb_id", lambda s: sample_ids(s, 5)))
    .reset_index()
    .sort_values("count", ascending=False)
    .head(20)
)

# 3) log.csv: unique log_name values with counts and sample cmdb_ids (up to 5)
log_summary = (
    df_log.groupby("log_name")
    .agg(count=("log_name", "size"), sample_cmdb_ids=("cmdb_id", lambda s: sample_ids(s, 5)))
    .reset_index()
    .sort_values("count", ascending=False)
    .head(20)
)

# 4) Count rows inside incident window for each file
start = pd.to_datetime("2024-01-18 20:21:16", utc=True)
end = pd.to_datetime("2024-01-18 20:51:16", utc=True)

# parse timestamps (Unix seconds -> UTC) and compute counts
df_metric["ts"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)
df_trace["ts"] = pd.to_datetime(df_trace["timestamp"], unit="s", utc=True)
df_log["ts"] = pd.to_datetime(df_log["timestamp"], unit="s", utc=True)
df_error["ts"] = pd.to_datetime(df_error["timestamp"], unit="s", utc=True)

metric_window_count = df_metric[(df_metric["ts"] >= start) & (df_metric["ts"] <= end)].shape[0]
trace_window_count = df_trace[(df_trace["ts"] >= start) & (df_trace["ts"] <= end)].shape[0]
log_window_count = df_log[(df_log["ts"] >= start) & (df_log["ts"] <= end)].shape[0]
error_window_count = df_error[(df_error["ts"] >= start) & (df_error["ts"] <= end)].shape[0]

window_counts = pd.DataFrame(
    {
        "file": ["metric.csv", "trace.csv", "log.csv", "error_logs.csv"],
        "rows_in_window": [metric_window_count, trace_window_count, log_window_count, error_window_count],
    }
)

# Display compact outputs (limited rows as requested)
metric_summary, trace_summary, log_summary, window_counts

```
Out[1]:
```
# Summarize results as a single plain-English string (compact)
num_metric_pairs = df_metric.groupby(["cmdb_id", "kpi_name"]).ngroups
metric_sample = metric_summary.head(5)[["cmdb_id", "kpi_name", "p95", "count_total"]]
trace_rows_total = df_trace.shape[0]
trace_unique_names = df_trace["trace_name"].nunique() if trace_rows_total>0 else 0
log_unique_names = df_log["log_name"].nunique()
log_sample = log_summary.head(5)[["log_name", "count", "sample_cmdb_ids"]]
window_df = window_counts.set_index("file")["rows_in_window"].to_dict()

summary = (
    f"Summary:\n"
    f"- metric.csv: {num_metric_pairs} unique (cmdb_id, kpi_name) pairs in total. "
    f"Sample top 5 pairs (cmdb_id, kpi_name, p95, count):\n{metric_sample.to_string(index=False)}\n\n"
    f"- trace.csv: total rows = {trace_rows_total}; unique trace_name values = {trace_unique_names}. "
    f"(No trace_name summary rows were returned earlier if empty.)\n\n"
    f"- log.csv: unique log_name values = {log_unique_names}. Sample top entries (log_name, count, sample_cmdb_ids):\n{log_sample.to_string(index=False)}\n\n"
    f"- Rows inside incident window [2024-01-18 20:21:16 UTC .. 2024-01-18 20:51:16 UTC]:\n"
    f"  metric.csv = {window_df.get('metric.csv', 0)}, trace.csv = {window_df.get('trace.csv', 0)}, "
    f"log.csv = {window_df.get('log.csv', 0)}, error_logs.csv = {window_df.get('error_logs.csv', 0)}\n\n"
    f"Notes: trace.csv had no summarized trace_name rows returned earlier; metric.csv summary shown was limited to top 50 pairs."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name           p95  count_total
0          carts         cpu  1.485806e+00           25
1          carts       error  0.000000e+00           25
2          carts  latency-50  3.628450e-02           25
3          carts  latency-90  1.246223e-01           25
4          carts         mem  2.167676e+08           25
..           ...         ...           ...          ...
45       payment    workload  2.155817e+00           25
46  queue-master         cpu  3.069094e-01           25
47  queue-master      diskio  4.023267e+04           25
48  queue-master         mem  2.674703e+08           25
49  queue-master      socket  3.000000e+00           25

[50 rows x 4 columns], Empty DataFrame
Columns: [trace_name, count, sample_cmdb_ids]
Index: [],           log_name  count                            sample_cmdb_ids
0  log.error_count    227  carts,carts-db,catalogue,front-end,orders
1    log.row_count    227  carts,carts-db,catalogue,front-end,orders,              file  rows_in_window
0      metric.csv            1925
1       trace.csv               0
2         log.csv             454
3  error_logs.csv               0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing df_metric if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric["ts"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)

# Ensure ts exists and is parsed
if "ts" not in df_metric.columns:
    df_metric["ts"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)

# 1) Define incident window and filter metric rows to that window
start = pd.to_datetime("2024-01-18 20:21:16", utc=True)
end = pd.to_datetime("2024-01-18 20:51:16", utc=True)
window_df = df_metric[(df_metric["ts"] >= start) & (df_metric["ts"] <= end)].copy()

# 2) Compute global P95 and P05 per (cmdb_id, kpi_name) using the full metric series
thresholds = (
    df_metric
    .groupby(["cmdb_id", "kpi_name"])["value"]
    .agg(p95=lambda x: float(x.quantile(0.95)), p05=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# 3) Merge thresholds into windowed data (so thresholds are global, not recomputed on window)
merged = window_df.merge(thresholds, on=["cmdb_id", "kpi_name"], how="left")

# 3) For each (cmdb_id, kpi_name) in the window compute required metrics
def summarize_group(g):
    p95 = g["p95"].iloc[0]
    p05 = g["p05"].iloc[0]
    count_in_window = int(g.shape[0])
    mask_high = g["value"] >= p95
    count_high_anoms = int(mask_high.sum())
    first_high_ts = g.loc[mask_high, "ts"].min() if count_high_anoms > 0 else pd.NaT
    mask_low = g["value"] <= p05
    count_low_anoms = int(mask_low.sum())
    first_low_ts = g.loc[mask_low, "ts"].min() if count_low_anoms > 0 else pd.NaT
    max_in_window = float(g["value"].max())
    min_in_window = float(g["value"].min())
    return pd.Series({
        "p95": p95,
        "p05": p05,
        "count_in_window": count_in_window,
        "count_high_anoms": count_high_anoms,
        "first_high_ts": first_high_ts,
        "count_low_anoms": count_low_anoms,
        "first_low_ts": first_low_ts,
        "max_in_window": max_in_window,
        "min_in_window": min_in_window,
    })

if merged.shape[0] > 0:
    grouped = merged.groupby(["cmdb_id", "kpi_name"]).apply(summarize_group).reset_index()
else:
    # empty structure if no rows in window
    grouped = pd.DataFrame(columns=[
        "cmdb_id","kpi_name","p95","p05","count_in_window","count_high_anoms",
        "first_high_ts","count_low_anoms","first_low_ts","max_in_window","min_in_window"
    ])

# 4) Filter to rows with any anomalies and sort as requested
if not grouped.empty:
    grouped["total_anoms"] = grouped["count_high_anoms"] + grouped["count_low_anoms"]
    anomalies = (
        grouped[(grouped["count_high_anoms"] > 0) | (grouped["count_low_anoms"] > 0)]
        .sort_values(["total_anoms", "cmdb_id"], ascending=[False, True])
        .drop(columns=["total_anoms"])
    )
else:
    anomalies = grouped.copy()

# Keep only requested columns and limit to top 20 rows
out_cols = [
    "cmdb_id","kpi_name","p95","p05","count_in_window",
    "count_high_anoms","first_high_ts","count_low_anoms","first_low_ts",
    "max_in_window","min_in_window"
]
anomalies_report = anomalies.loc[:, out_cols].head(20)

# 5) One-line count of distinct (cmdb_id,kpi_name) pairs that had any anomaly in-window
anomaly_pairs_count = anomalies.shape[0]

# Return compact results
anomalies_report, anomaly_pairs_count

```
Out[2]:
```
summary = (
    "Summary of metric anomaly scan (window 2024-01-18 20:21:16 → 20:51:16 UTC):\n"
    "- Total distinct (cmdb_id, kpi_name) pairs with any anomaly in-window: 77.\n"
    "- Most affected components (examples from top results): carts, catalogue, front-end, orders, rabbitmq, rabbitmq-exporter, session-db, user-db, payment, queue-master, shipping.\n"
    "- Most common KPI types showing anomalies: socket, error, diskio, mem, and latency-50.\n"
    "- Many anomalies are persistent across the window (count_in_window = 25 for many pairs), indicating the issue lasted the whole 30-minute window for those series.\n"
    "- Earliest anomalous samples are concentrated at 2024-01-18 20:24:00 UTC for many series; a few series show first high anomalies later (e.g., payment first high at 20:37:00, shipping high at 20:43:00).\n"
    "- Notable patterns: several services have both high and low threshold breaches (e.g., socket and error KPIs), and disk I/O and socket metrics are repeatedly implicated.\n\n"
    "Conclusion: The incident affected many services and KPI types simultaneously, with socket and error metrics particularly prominent and anomalies peaking/starting around 20:24:00 UTC. Investigate network/socket-related issues and disk I/O on the identified services first."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name           p95          p05  count_in_window  count_high_anoms             first_high_ts  count_low_anoms              first_low_ts  max_in_window  min_in_window
1               carts       error  0.000000e+00        0.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   0.000000e+00          0.000
10           carts-db      socket  1.000000e+01       10.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   1.000000e+01         10.000
12          catalogue       error  0.000000e+00        0.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   0.000000e+00          0.000
16          catalogue      socket  7.000000e+00        7.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   7.000000e+00          7.000
19       catalogue-db      diskio  0.000000e+00        0.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   0.000000e+00          0.000
21       catalogue-db      socket  4.000000e+00        4.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   4.000000e+00          4.000
23          front-end       error  0.000000e+00        0.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   0.000000e+00          0.000
30             orders       error  0.000000e+00        0.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   0.000000e+00          0.000
39          orders-db      socket  6.000000e+00        6.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   6.000000e+00          6.000
49       queue-master      socket  3.000000e+00        3.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   3.000000e+00          3.000
51           rabbitmq      diskio  0.000000e+00        0.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   0.000000e+00          0.000
53           rabbitmq      socket  1.100000e+01       11.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   1.100000e+01         11.000
55  rabbitmq-exporter         mem  4.575232e+06  4575232.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   4.575232e+06    4575232.000
56  rabbitmq-exporter      socket  1.000000e+00        1.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   1.000000e+00          1.000
60         session-db      socket  3.000000e+00        3.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   3.000000e+00          3.000
76            user-db      socket  8.000000e+00        8.000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00   8.000000e+00          8.000
44            payment      socket  7.000000e+00        5.000               25                12 2024-01-18 20:37:00+00:00               12 2024-01-18 20:24:00+00:00   7.000000e+00          5.000
71               user      socket  1.800000e+01       17.000               25                 8 2024-01-18 20:24:00+00:00               16 2024-01-18 20:33:00+00:00   1.800000e+01         17.000
58         session-db      diskio  1.428064e+04        0.000               25                 2 2024-01-18 20:25:00+00:00               15 2024-01-18 20:24:00+00:00   1.627345e+04          0.000
62           shipping  latency-50  3.052415e-03        0.003               25                 2 2024-01-18 20:43:00+00:00                6 2024-01-18 20:25:00+00:00   3.089423e-03          0.003, 77)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure timestamps parsed
if "ts" not in df_metric.columns:
    df_metric["ts"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)
if "ts" not in df_log.columns:
    df_log["ts"] = pd.to_datetime(df_log["timestamp"], unit="s", utc=True)

# Candidate services
candidates = ["carts", "catalogue", "orders", "payment", "user"]

# Incident window
start = pd.to_datetime("2024-01-18 20:21:16", utc=True)
end = pd.to_datetime("2024-01-18 20:51:16", utc=True)

# 1) Compute global thresholds (P95, P05) on full metric series BEFORE filtering
thresholds = (
    df_metric
    .groupby(["cmdb_id", "kpi_name"])["value"]
    .agg(p95=lambda x: float(x.quantile(0.95)), p05=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# Filter thresholds to candidate services
thresholds_cand = thresholds[thresholds["cmdb_id"].isin(candidates)].copy()

# Filter metric rows to incident window and candidate services
window_metric = df_metric[
    (df_metric["ts"] >= start) & (df_metric["ts"] <= end) & (df_metric["cmdb_id"].isin(candidates))
].copy()

# Merge global thresholds into the windowed metric rows
merged = window_metric.merge(thresholds_cand, on=["cmdb_id", "kpi_name"], how="left")

# Summarize per (cmdb_id, kpi_name)
def summarize_metric_group(g):
    p95 = g["p95"].iloc[0]
    p05 = g["p05"].iloc[0]
    count_in_window = int(g.shape[0])
    mask_high = g["value"] >= p95
    count_high_anoms = int(mask_high.sum())
    first_high_ts = g.loc[mask_high, "ts"].min() if count_high_anoms > 0 else pd.NaT
    mask_low = g["value"] <= p05
    count_low_anoms = int(mask_low.sum())
    first_low_ts = g.loc[mask_low, "ts"].min() if count_low_anoms > 0 else pd.NaT
    max_in_window = float(g["value"].max())
    min_in_window = float(g["value"].min())
    # severity: NaN if p95 == 0 else (max_in_window - p95)/max(1e-9, p95)
    severity = np.nan if p95 == 0 else (max_in_window - p95) / max(1e-9, p95)
    return pd.Series({
        "p95": p95,
        "p05": p05,
        "count_in_window": count_in_window,
        "count_high_anoms": count_high_anoms,
        "first_high_ts": first_high_ts,
        "count_low_anoms": count_low_anoms,
        "first_low_ts": first_low_ts,
        "max_in_window": max_in_window,
        "min_in_window": min_in_window,
        "severity": severity
    })

if not merged.empty:
    metric_grouped = merged.groupby(["cmdb_id", "kpi_name"]).apply(summarize_metric_group).reset_index()
else:
    metric_grouped = pd.DataFrame(columns=[
        "cmdb_id","kpi_name","p95","p05","count_in_window","count_high_anoms","first_high_ts",
        "count_low_anoms","first_low_ts","max_in_window","min_in_window","severity"
    ])

# Filter to rows with any anomalies and sort by (count_high_anoms+count_low_anoms) desc then severity desc
if not metric_grouped.empty:
    metric_grouped["total_anoms"] = metric_grouped["count_high_anoms"] + metric_grouped["count_low_anoms"]
    metric_fault_table = (
        metric_grouped[(metric_grouped["count_high_anoms"] > 0) | (metric_grouped["count_low_anoms"] > 0)]
        .sort_values(["total_anoms", "severity"], ascending=[False, False])
        .drop(columns=["total_anoms"])
    )
else:
    metric_fault_table = metric_grouped.copy()

# Limit to top 20 rows for compactness
metric_fault_table = metric_fault_table.loc[:, [
    "cmdb_id","kpi_name","p95","p05","count_in_window","count_high_anoms","first_high_ts",
    "count_low_anoms","first_low_ts","max_in_window","min_in_window","severity"
]].head(20)

# 2) Log summary for same window and candidate services
log_window = df_log[(df_log["ts"] >= start) & (df_log["ts"] <= end) & (df_log["cmdb_id"].isin(candidates))].copy()

# Prepare aggregations
if not log_window.empty:
    # Sum values per cmdb_id & log_name, then pivot
    agg = (
        log_window.groupby(["cmdb_id", "log_name"])["value"]
        .sum()
        .reset_index()
        .pivot(index="cmdb_id", columns="log_name", values="value")
        .reset_index()
    )
    # Count of log rows per cmdb_id (in window)
    row_counts = log_window.groupby("cmdb_id").size().rename("log_row_count").reset_index()
    # earliest timestamp where log.error_count > 0
    errs = log_window[(log_window["log_name"] == "log.error_count") & (log_window["value"] > 0)]
    first_err_ts = errs.groupby("cmdb_id")["ts"].min().rename("first_nonzero_error_ts").reset_index()
    # Merge together
    log_summary = pd.merge(pd.DataFrame({"cmdb_id": candidates}), agg, on="cmdb_id", how="left")
    log_summary = pd.merge(log_summary, row_counts, on="cmdb_id", how="left")
    log_summary = pd.merge(log_summary, first_err_ts, on="cmdb_id", how="left")
else:
    # create empty structure with candidate rows
    log_summary = pd.DataFrame({"cmdb_id": candidates})
    log_summary["log.error_count"] = np.nan
    log_summary["log.row_count"] = np.nan
    log_summary["log_row_count"] = 0
    log_summary["first_nonzero_error_ts"] = pd.NaT

# Normalize column names and fill missing sums with 0 per instruction
log_summary = log_summary.rename(columns={
    "log.error_count": "sum_log_error_count",
    "log.row_count": "sum_log_row_count"
})
log_summary["sum_log_error_count"] = log_summary.get("sum_log_error_count", pd.Series()).fillna(0)
log_summary["sum_log_row_count"] = log_summary.get("sum_log_row_count", pd.Series()).fillna(0)
log_summary["log_row_count"] = log_summary.get("log_row_count", pd.Series()).fillna(0).astype(int)
# Ensure first_nonzero_error_ts exists
if "first_nonzero_error_ts" not in log_summary.columns:
    log_summary["first_nonzero_error_ts"] = pd.NaT

# Reorder columns as requested
log_summary = log_summary[[
    "cmdb_id", "sum_log_error_count", "sum_log_row_count", "log_row_count", "first_nonzero_error_ts"
]]

# Limit rows (we only have candidate services so this is already small)
log_summary = log_summary.head(20)

# Return compact tables
metric_fault_table, log_summary

```
Out[3]:
```
summary = (
    "Summary for candidates (carts, catalogue, orders, payment, user) during 2024-01-18 20:21:16→20:51:16 UTC:\n"
    "- Metric anomalies: All five services show metric anomalies in the window.  The most prominent KPI types are 'socket' and 'error' (many series had count_in_window=25, i.e. persistent for the whole window).  Several anomalies first appear around 2024-01-18 20:24:00 UTC.\n"
    "- Notable specifics: catalogue.socket and multiple services' error metrics are anomalous for the entire window. payment and user show repeated socket breaches (payment has high anomalies starting later at ~20:37 for some points). CPU and latency anomalies exist but with much smaller severity values than socket/error breaches.\n"
    "- Severity: Many series have low numeric severity (small relative increases above P95); some KPIs have p95==0 (severity NaN) where value==0 anomalies were detected (error metrics).\n"
    "- Logs: Aggregated log data for these services in the same window show no nonzero log.error_count (first_nonzero_error_ts is empty/NaT for all candidates), though log row counts are present—so logs do not report explicit errors even while metrics show socket/error anomalies.\n"
    "Conclusion: The telemetry points to a cross-service socket/network-related problem (and error counters rising) starting ~20:24 UTC, affecting carts, catalogue, orders, payment, and user. Investigate network/socket layer and any upstream networking components (load balancer, service mesh, or host networking) first; correlate with infrastructure/network telemetry and trace data next."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name        p95        p05  count_in_window  count_high_anoms             first_high_ts  count_low_anoms              first_low_ts  max_in_window  min_in_window  severity
12  catalogue      socket   7.000000   7.000000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00       7.000000       7.000000  0.000000
1       carts       error   0.000000   0.000000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00       0.000000       0.000000       NaN
8   catalogue       error   0.000000   0.000000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00       0.000000       0.000000       NaN
15     orders       error   0.000000   0.000000               25                25 2024-01-18 20:24:00+00:00               25 2024-01-18 20:24:00+00:00       0.000000       0.000000       NaN
25    payment      socket   7.000000   5.000000               25                12 2024-01-18 20:37:00+00:00               12 2024-01-18 20:24:00+00:00       7.000000       5.000000  0.000000
31       user      socket  18.000000  17.000000               25                 8 2024-01-18 20:24:00+00:00               16 2024-01-18 20:33:00+00:00      18.000000      17.000000  0.000000
3       carts  latency-90   0.124622   0.089281               25                 2 2024-01-18 20:29:00+00:00                2 2024-01-18 20:26:00+00:00       0.147158       0.087005  0.180828
19     orders      socket  16.383333  14.550000               25                 2 2024-01-18 20:44:00+00:00                2 2024-01-18 20:29:00+00:00      17.983333      14.000000  0.097660
22    payment  latency-50   0.003135   0.001820               25                 2 2024-01-18 20:37:00+00:00                2 2024-01-18 20:29:00+00:00       0.003417       0.001773  0.089877
14     orders         cpu   1.165657   0.915608               25                 2 2024-01-18 20:25:00+00:00                2 2024-01-18 20:41:00+00:00       1.205542       0.894602  0.034217
28       user  latency-50   0.004660   0.004302               25                 2 2024-01-18 20:32:00+00:00                2 2024-01-18 20:28:00+00:00       0.004808       0.004191  0.031745
0       carts         cpu   1.485806   1.288038               25                 2 2024-01-18 20:31:00+00:00                2 2024-01-18 20:39:00+00:00       1.530892       1.264222  0.030345
23    payment  latency-90   0.063153   0.004364               25                 2 2024-01-18 20:40:00+00:00                2 2024-01-18 20:29:00+00:00       0.064994       0.004355  0.029140
17     orders  latency-90   0.120601   0.047932               25                 2 2024-01-18 20:40:00+00:00                2 2024-01-18 20:24:00+00:00       0.123525       0.047796  0.024242
2       carts  latency-50   0.036284   0.031161               25                 2 2024-01-18 20:41:00+00:00                2 2024-01-18 20:28:00+00:00       0.037005       0.030588  0.019867
27       user         cpu   1.057063   0.948375               25                 2 2024-01-18 20:36:00+00:00                2 2024-01-18 20:26:00+00:00       1.076287       0.938418  0.018187
26    payment    workload   2.155817   1.923260               25                 2 2024-01-18 20:31:00+00:00                2 2024-01-18 20:41:00+00:00       2.182267       1.898783  0.012269
7   catalogue         cpu   0.182531   0.167629               25                 2 2024-01-18 20:25:00+00:00                2 2024-01-18 20:41:00+00:00       0.184570       0.163748  0.011167
20     orders    workload   2.133247   1.947847               25                 2 2024-01-18 20:36:00+00:00                2 2024-01-18 20:39:00+00:00       2.156717       1.889917  0.011002
13  catalogue    workload   4.219923   3.884147               25                 2 2024-01-18 20:40:00+00:00                2 2024-01-18 20:27:00+00:00       4.258733       3.829850  0.009197,      cmdb_id  sum_log_error_count  sum_log_row_count  log_row_count first_nonzero_error_ts
0      carts                    0               1575             46                    NaT
1  catalogue                    0               3900             50                    NaT
2     orders                    0               2938             50                    NaT
3    payment                    0               2429             50                    NaT
4       user                    0              14187             50                    NaT)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timestamp parsed to UTC datetime
if "ts" not in df_metric.columns:
    df_metric["ts"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)

# Candidate services
candidates = ["carts", "catalogue", "orders", "payment", "user"]

# Incident window
start = pd.to_datetime("2024-01-18 20:21:16", utc=True)
end = pd.to_datetime("2024-01-18 20:51:16", utc=True)

# 1+2) Compute global thresholds (P95, P05) on full metric series BEFORE filtering
thresholds = (
    df_metric
    .groupby(["cmdb_id", "kpi_name"])["value"]
    .agg(p95=lambda x: float(x.quantile(0.95)), p05=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# Filter thresholds to candidate services
thresholds_cand = thresholds[thresholds["cmdb_id"].isin(candidates)].copy()

# Filter metric rows to the incident window and candidate services
window_metric = df_metric[
    (df_metric["ts"] >= start) & (df_metric["ts"] <= end) & (df_metric["cmdb_id"].isin(candidates))
].copy()

# Merge global thresholds into the windowed metric rows
merged = window_metric.merge(thresholds_cand, on=["cmdb_id", "kpi_name"], how="left")

# 2) Per (cmdb_id,kpi_name) compute anomaly counts, earliest ts, severity
def summarize_kpi(g):
    p95 = g["p95"].iloc[0]
    p05 = g["p05"].iloc[0]
    count_in_window = int(g.shape[0])
    mask_high = g["value"] >= p95
    mask_low = g["value"] <= p05
    count_high = int(mask_high.sum())
    count_low = int(mask_low.sum())
    first_high = g.loc[mask_high, "ts"].min() if count_high > 0 else pd.NaT
    first_low = g.loc[mask_low, "ts"].min() if count_low > 0 else pd.NaT
    # earliest anomaly timestamp for this KPI
    if pd.isna(first_high) and pd.isna(first_low):
        earliest_anom = pd.NaT
    elif pd.isna(first_high):
        earliest_anom = first_low
    elif pd.isna(first_low):
        earliest_anom = first_high
    else:
        earliest_anom = min(first_high, first_low)
    max_in_window = float(g["value"].max())
    # severity: 0 when p95==0 or NaN, else (max_in_window - p95)/p95
    if (p95 == 0) or pd.isna(p95):
        severity = 0.0
    else:
        severity = float((max_in_window - p95) / p95)
    any_anom = (count_high > 0) or (count_low > 0)
    return pd.Series({
        "p95": p95,
        "p05": p05,
        "count_in_window": count_in_window,
        "count_high_anoms": count_high,
        "first_high_ts": first_high,
        "count_low_anoms": count_low,
        "first_low_ts": first_low,
        "earliest_anom_ts": earliest_anom,
        "max_in_window": max_in_window,
        "severity": severity,
        "any_anom": any_anom
    })

if not merged.empty:
    kpi_summary = merged.groupby(["cmdb_id", "kpi_name"]).apply(summarize_kpi).reset_index()
else:
    kpi_summary = pd.DataFrame(columns=[
        "cmdb_id","kpi_name","p95","p05","count_in_window","count_high_anoms","first_high_ts",
        "count_low_anoms","first_low_ts","earliest_anom_ts","max_in_window","severity","any_anom"
    ])

# Filter to KPIs that had any anomaly in-window
kpi_anoms = kpi_summary[kpi_summary["any_anom"]].copy()

# 3) Aggregate per cmdb_id (service)
if not kpi_anoms.empty:
    svc_agg = kpi_anoms.groupby("cmdb_id").agg(
        distinct_kpis_with_any_anomaly=("kpi_name", "nunique"),
        total_anomalous_points=("count_high_anoms", "sum"),  # we'll add low counts next
        earliest_anomaly_ts=("earliest_anom_ts", "min"),
        max_kpi_severity=("severity", "max"),
        sum_kpi_severity=("severity", "sum")
    ).reset_index()
    # add low anomaly counts into total_anomalous_points
    low_counts = kpi_anoms.groupby("cmdb_id")["count_low_anoms"].sum().rename("sum_low")
    svc_agg = svc_agg.merge(low_counts.reset_index(), on="cmdb_id", how="left")
    svc_agg["total_anomalous_points"] = svc_agg["total_anomalous_points"] + svc_agg["sum_low"]
    svc_agg = svc_agg.drop(columns=["sum_low"])
else:
    # empty structure with candidate services (fill zeros/NaT)
    svc_agg = pd.DataFrame({
        "cmdb_id": candidates,
        "distinct_kpis_with_any_anomaly": 0,
        "total_anomalous_points": 0,
        "earliest_anomaly_ts": pd.NaT,
        "max_kpi_severity": 0.0,
        "sum_kpi_severity": 0.0
    })

# Ensure all candidate services present (including those with zero anomalies)
svc_all = pd.DataFrame({"cmdb_id": candidates})
svc_summary = svc_all.merge(svc_agg, on="cmdb_id", how="left").fillna({
    "distinct_kpis_with_any_anomaly": 0,
    "total_anomalous_points": 0,
    "earliest_anomaly_ts": pd.NaT,
    "max_kpi_severity": 0.0,
    "sum_kpi_severity": 0.0
})
# cast counts to int
svc_summary["distinct_kpis_with_any_anomaly"] = svc_summary["distinct_kpis_with_any_anomaly"].astype(int)
svc_summary["total_anomalous_points"] = svc_summary["total_anomalous_points"].astype(int)

# 4) Sort as requested and limit to 5 rows
svc_summary = svc_summary.sort_values(
    ["distinct_kpis_with_any_anomaly", "total_anomalous_points"], ascending=[False, False]
).reset_index(drop=True).head(5)

# Top candidate (first row by the ordering)
top_candidate = svc_summary["cmdb_id"].iloc[0] if not svc_summary.empty else None
top_candidate_str = f"Top candidate service: {top_candidate}" if top_candidate is not None else "No anomalies detected"

# Return compact results
svc_summary, top_candidate_str

```
Out[4]:
```
# Compose a concise plain-English summary string based on previous results
try:
    df = svc_summary.copy()
except NameError:
    df = None

if df is None or df.empty:
    summary = "No anomaly summary available."
else:
    # extract rows for message
    def fmt(v, digits=6):
        if pd.isna(v):
            return "NaN"
        if isinstance(v, (float, int)):
            return f"{v:.6g}"
        return str(v)

    top = df.iloc[0]
    rows = []
    for i, r in df.iterrows():
        rows.append(
            f"{i+1}) {r['cmdb_id']}: {r['distinct_kpis_with_any_anomaly']} KPIs, "
            f"{r['total_anomalous_points']} anomalous points, earliest {r['earliest_anomaly_ts']}, "
            f"max_severity={fmt(r['max_kpi_severity'],6)}, sum_severity={fmt(r['sum_kpi_severity'],6)}"
        )
    ranking = "\n".join(rows)

    summary = (
        f"Summary of anomalies for candidates within 2024-01-18 20:21:16→20:51:16 UTC:\n"
        f"- Top service: {top['cmdb_id']} (rank 1). It has {top['distinct_kpis_with_any_anomaly']} distinct KPIs with anomalies, "
        f"{top['total_anomalous_points']} total anomalous points, earliest anomaly at {top['earliest_anomaly_ts']}, "
        f"max KPI severity {fmt(top['max_kpi_severity'],6)}, sum KPI severity {fmt(top['sum_kpi_severity'],6)}.\n\n"
        f"- Full ranking (by distinct KPIs then anomalous points):\n{ranking}\n\n"
        f"All five services show anomalies starting around 2024-01-18 20:24:00 UTC. Investigate {top['cmdb_id']} first, then the other top-ranked services (carts, orders, payment, user)."
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  distinct_kpis_with_any_anomaly  total_anomalous_points       earliest_anomaly_ts  max_kpi_severity  sum_kpi_severity
0  catalogue                               7                     120 2024-01-18 20:24:00+00:00          0.011167          0.028357
1      carts                               7                      74 2024-01-18 20:24:00+00:00          0.180828          0.246389
2     orders                               7                      74 2024-01-18 20:24:00+00:00          0.097660          0.173962
3    payment                               6                      44 2024-01-18 20:24:00+00:00          0.089877          0.133156
4       user                               6                      44 2024-01-18 20:24:00+00:00          0.031745          0.060163, 'Top candidate service: catalogue')```
```